# Appendix — Google ADK: agent, runner, and session

Seven framework appendices run the *same* Larkspur triage — ticket `TKT-2205`, gold `partial_refund | pol-restocking | $170.99` — so you can compare frameworks on one fixed problem. This is the **agent-runtime** archetype: [Google ADK](https://adk.dev/) is "the open-source agent development framework that lets you build, debug, and deploy reliable AI agents," and its distinguishing move is the split between an `LlmAgent` (the reasoning unit) and a `Runner` that drives it against a `Session`. That is your chapter 02 tool loop with the loop, the tool registry, and the message history each promoted to a named object.

> **Before running this notebook:** `pip install -e ".[adk]"` (once). It pulls in `google-adk` (imported as `google.adk`), which co-installs with the main venv — no separate kernel. The model reaches the same OpenRouter endpoint you have used since ch01 through ADK's `LiteLlm` wrapper, so the disk cache from the config cell still sees these calls. Everything else stays the same.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The model boundary, as a LiteLlm wrapper

ADK is model-agnostic: an agent reaches its model through a *model object*, not `shoplab.llm.complete`. For any non-Google provider that object is `LiteLlm`, a thin adapter over the same LiteLLM you have called since ch01 — so the OpenRouter routing prefix and `OPENROUTER_API_KEY` are unchanged and the model string is just `MODEL`. Because it *is* LiteLLM underneath, `temperature=0` and the disk cache both carry over; a rerun is ~free.

In [ ]:
from google.adk.models.lite_llm import LiteLlm

model = LiteLlm(model=MODEL, temperature=TEMPERATURE)   # same endpoint as ch01
print("model wrapper:", type(model).__name__, "->", model.model)

## The gold invariant

Every appendix must land on the same economics, so pin the invariant first — the authoritative gold from `shoplab.rules.decide` for `TKT-2205` (an opened, in-window return from a non-vip *member*: `$189.99 x 0.90`). The agent below must reproduce this number by reasoning and tool use, not by being told it.

In [ ]:
from shoplab import world, rules

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
t2205 = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
GOLD = rules.decide(t2205, orders[t2205["order_id"]], customers[t2205["customer_id"]])
print("authoritative gold (shoplab.rules.decide):", GOLD)   # the invariant every appendix hits

## Tools: thin wrappers over shoplab

An `LlmAgent` takes a list of tools. A plain annotated Python function *is* a tool — ADK reads the signature and docstring into a JSON schema, exactly what `shoplab.tools.to_openai_tools` did by hand in ch02. The four read-only lookups (`get_order`, `get_customer`, `search_policy`, `calc`) are thin wrappers that *import* from `shoplab.world` / `shoplab.tools`, not reimplementations; `finish` is ch02's terminator, which hands back a structured decision (ADK cannot combine an `output_schema` with tools, so the verdict comes through a tool call, not a typed return). The risky `issue_refund` is wrapped in a `FunctionTool(..., require_confirmation=True)` — the flag that becomes the approval gate two cells down.

In [ ]:
from typing import Optional
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from shoplab.tools import calc as _calc, Ledger

ledger = Ledger()          # ch02's real side-effect log
verdict = {}               # finish writes the structured decision here

def get_order(order_id: str) -> dict:
    """Look up a Larkspur order by id (items, totals, status, dates)."""
    return orders.get(order_id, {"error": f"no such order {order_id}"})

def get_customer(customer_id: str) -> dict:
    """Look up a Larkspur customer by id (tier, flags, history)."""
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

def search_policy(query: str, k: int = 2) -> list:
    """Keyword-search the 12 Larkspur store policy documents."""
    return world.search_policy(query, k=k)

def calc(expr: str) -> float:
    """Evaluate an arithmetic expression, e.g. '0.9 * 189.99'."""
    return _calc(expr)

def issue_refund(order_id: str, amount_usd: float, reason: str) -> dict:
    """Send money back to the customer. Irreversible."""
    entry = ledger.record("issue_refund", order_id=order_id,
                          amount_usd=amount_usd, reason=reason)
    return {"ok": True, "refund_id": f"REF-{1000 + len(ledger.entries)}", **entry}

def finish(decision: str, policy_id: str, refund_usd: Optional[float] = None) -> dict:
    """Record the final ticket decision and stop. decision is one of approve_refund,
    partial_refund, replacement, store_credit, deny, escalate; refund_usd is the dollars
    moved, normalized to cents (models sometimes send it as a string or an extra digit)."""
    amount = round(float(refund_usd), 2) if refund_usd is not None else None
    verdict.update(decision=decision, policy_id=policy_id, refund_usd=amount)
    return {"recorded": True}

INSTRUCTION = (
    "You are the Larkspur Outfitters ops desk. Triage the return ticket: look up the "
    "order and the customer, search the governing policy, compute any restocking fee "
    "with calc, then call issue_refund to move the money. Opened non-vip returns carry "
    "a 10 percent restocking fee. Round every dollar amount to the nearest cent (two "
    "decimals). End by calling finish with the decision (one of approve_refund, "
    "partial_refund, replacement, store_credit, deny, escalate), the policy_id, and "
    "the refund_usd amount.")

agent = LlmAgent(name="ops_desk", model=model, instruction=INSTRUCTION,
                 tools=[get_order, get_customer, search_policy, calc,
                        FunctionTool(issue_refund, require_confirmation=True), finish])
runner = InMemoryRunner(agent=agent, app_name="larkspur")
print("wired: get_order, get_customer, search_policy, calc, finish "
      "+ issue_refund (risky, confirmation-gated)")

## The approval gate is a tool confirmation

Chapter 08 wrapped the risky tools in `require_approval` so money moved only after a human said yes. ADK ships that as `require_confirmation=True`. When the model calls `issue_refund`, ADK does **not** run it: it emits an `adk_request_confirmation` call (a *long-running* tool the runtime cannot resolve itself) and the run stream ends, paused, with the pending call id in `event.long_running_tool_ids`. We drive the `Runner` by iterating its event stream — the ADK analog of stepping ch02's while-loop — and stop at that pause. The `Ledger` stays empty.

In [ ]:
from google.genai import types

TICKET = ("Triage ticket TKT-2205 (order ORD-7312, customer CUST-07, sku LK-1016, qty 1): "
          "'I opened the box and used the Torrent boots one evening indoors, they pinch at "
          "the toes. Repacked with tags. Refund my original payment method.'")

def confirmation_id(event):
    """The id of a pending adk_request_confirmation call on this event, or None."""
    pending = getattr(event, "long_running_tool_ids", None) or set()
    for part in (event.content.parts if event.content else []):
        fc = part.function_call
        if fc and fc.name == "adk_request_confirmation" and fc.id in pending:
            return fc.id
    return None

async def run_to_pause():
    session = await runner.session_service.create_session(app_name="larkspur", user_id="ops")
    interrupt_id = None
    async for event in runner.run_async(user_id="ops", session_id=session.id,
            new_message=types.Content(role="user", parts=[types.Part(text=TICKET)])):
        for fc in event.get_function_calls():
            print("tool call:", fc.name)
        interrupt_id = confirmation_id(event) or interrupt_id
    return session.id, interrupt_id

SESSION_ID, interrupt_id = await run_to_pause()             # multi-step loop, then the gate
print("paused for confirmation:", interrupt_id is not None)
print("ledger entries:", len(ledger.entries), "(gate held -- no money moved yet)")

> **What you should see:** the tool calls stream by in order — `get_order`, `get_customer`, `search_policy`, `calc` (the `0.90 x 189.99` fee) — and then the run stops with `adk_request_confirmation` pending, so `interrupt_id` is set. The `Ledger` still reads `0`: `require_confirmation=True` turned the `issue_refund` call into a pause, not a side effect. This is chapter 08's approval gate, handed to you by the framework.

In [ ]:
async def approve_and_resume(interrupt_id):
    # A human reviews the pending call and approves (ch08's approver returns True).
    approval = types.Content(role="user", parts=[types.Part(
        function_response=types.FunctionResponse(
            id=interrupt_id, name="adk_request_confirmation",
            response={"confirmed": True}))])
    async for event in runner.run_async(user_id="ops", session_id=SESSION_ID,
                                        new_message=approval):
        for fc in event.get_function_calls():
            print("tool call:", fc.name)

await approve_and_resume(interrupt_id)                      # same session, resumed
print("ledger entries:", len(ledger.entries), "(the approved write happened)")
print("verdict        :", verdict)
print("matches gold   :", verdict == GOLD)

> **What you should see:** resuming the *same* `Session` with the confirmation turns the pending call into a real one — `issue_refund` runs, the `Ledger` ticks to `1`, and the model calls `finish`. `verdict` reads `partial_refund | pol-restocking | 170.99` and `matches gold` is `True`: ADK landed on the same economics as `shoplab.rules.decide` (`$189.99 x 0.90`), the invariant every appendix in this set shares. The session carried the whole transcript across the pause; you supplied only the one-line approval.

## Machinery map: ADK feature to the part you built

Line them up and the framework stops being magic. Every ADK concept here maps to a piece of machinery you built by hand in Parts 1-3.

| ADK concept | Your hand-built equivalent | Built in |
|---|---|---|
| `LlmAgent` + `Runner` driving the event stream | `run_agent`'s while-loop: model call -> tool call -> observe, repeat | ch02 |
| plain functions / `FunctionTool` (signature+docstring -> schema) | the `Tool` registry and `to_openai_tools` building the schema by hand | ch02 |
| `finish` as a tool the model calls to stop | the `finish` terminator ending the loop with a decision | ch02 |
| `LiteLlm(model=MODEL)` | `shoplab.llm.complete` wrapping the model boundary | ch01 |
| `Session` + `InMemoryRunner` session service | the `messages` list threaded through the loop by hand | ch02 |
| `require_confirmation=True` + `adk_request_confirmation` pause | `require_approval` wrapping a risky tool in a yes/no gate | ch08 |
| a `FunctionResponse` with `confirmed=True` to resume | the approver returning `True` and the loop continuing | ch08 |

## The honest trade

ADK's bet is a runtime. You never wrote the stepping loop here: the `Runner` drove the `LlmAgent` against a `Session` that accumulated the transcript, and `require_confirmation=True` reproduced chapter 08's pause-inspect-resume as an event in that stream plus a one-line `FunctionResponse` to continue. The session is real persistence — the pause and resume are two separate `run_async` calls over the same stored history, the seam chapter 12 built by hand.

The cost is ceremony and a moved seam. The verdict could not come back as a typed object the way a `output_schema` agent would give you, *because* this agent uses tools — ADK forbids the combination — so `finish` carries the decision instead, and you read it out of a tool call. The confirmation round-trip is `adk_request_confirmation` bookkeeping (match the pending id, echo it back), heavier than a plain `if approved:`. And, as chapters 11 and 16 warned, these calls flow through ADK's `LiteLlm`; the disk cache still sees them, but the cost `LEDGER` from earlier chapters does not. You are buying a session-backed runtime and a first-class approval pause; know the loop is now the framework's, not yours.